# 🔬 DPS Global Conditional — Main Notebook
Diffusion Posterior Sampling with MMD-based guidance using SDXL Turbo + ControlNet.

## 1. Environment Setup & GitHub Clone

In [ ]:
import os
import sys

# ── Colab: install deps & clone repo ─────────────────────────────────────────
if 'google.colab' in str(get_ipython()):
    import getpass

    # Install required packages
    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    # ── GitHub Auth ───────────────────────────────────────────────────────────
    # Option A: hidden prompt (recommended for shared notebooks)
    # token = getpass.getpass("Enter your GitHub personal access token: ")

    # Option B: hardcoded token (replace with your own — DO NOT commit to public repos)
    token = r"YOUR_GITHUB_PAT_HERE"

    repo_url  = f"https://{token}@github.com/orineo1/GlobalConditional.git"
    repo_name = "conditional-matching-paper"
    branch    = "master"

    # Clone if not already present
    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    # Checkout target branch
    !cd {repo_name} && git checkout {branch}
    # intall requirements
    !pip install -q -r {repo_name}/requirements.txt
    # Add repo to Python path
    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

# ── Local dev: just add local project root to path ────────────────────────────
else:
    LOCAL_ROOT = os.path.abspath(".")
    if LOCAL_ROOT not in sys.path:
        sys.path.insert(0, LOCAL_ROOT)
    print(f"🖥️  Running locally. Project root: {LOCAL_ROOT}")

## 2. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import gc
from sklearn.decomposition import PCA
from IPython.display import display

# ── Project modules (from cloned repo) ───────────────────────────────────────
from models       import load_models, setup_gradient_checkpointing
from image_utils  import sobel_proxy, latent_to_pil, build_base_image
from metrics      import compute_mmd
from generation   import (
    generate_and_store,
    predict_noise_cfg,
    compute_pred_x0,
    denoise_step,
    run_dps_step,
)
from visualization import plot_row, visualize_step

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

## 3. Load Models

In [ ]:
architect, sprinter = load_models(device)
print("✅ Models loaded.")

## 4. Build Base Image & Sobel Conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)

with torch.no_grad():
    sobel_cond_tensor = sobel_proxy(base_tensor, device)
    sobel_cond_pil    = T.ToPILImage()(sobel_cond_tensor.squeeze(0).cpu())

# Visual check
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(base_image_pil);  axes[0].set_title("Base Shape");       axes[0].axis('off')
axes[1].imshow(sobel_cond_pil, cmap='gray'); axes[1].set_title("Sobel Conditioning"); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 5. Generate Target Distributions

In [ ]:
N       = 20
n_total = N // 2

with torch.no_grad():
    mag_images, mag_latents = generate_and_store(
        sprinter,
        "a superrealistic magnifying glass single object",
        sobel_cond_pil, n_total, batch_size=2
    )
    pan_images, pan_latents = generate_and_store(
        sprinter,
        "a superrealistic pan single object",
        sobel_cond_pil, n_total, batch_size=2
    )

all_latents = np.vstack([mag_latents, pan_latents])
print(f"✅ Target latents shape: {all_latents.shape}")

## 6. Explore Target Samples + PCA

In [ ]:
plot_row(mag_images, "Magnifying Glass Samples")
plot_row(pan_images, "Frying Pan Samples")

# PCA of target latent space
pca_viz  = PCA(n_components=2)
coords   = pca_viz.fit_transform(all_latents)

plt.figure(figsize=(10, 7))
plt.scatter(coords[:n_total, 0], coords[:n_total, 1], c='dodgerblue', label='Magnifying Glass', alpha=0.6)
plt.scatter(coords[n_total:, 0], coords[n_total:, 1], c='crimson',    label='Frying Pan',      alpha=0.6)
plt.title("PCA of Target Latent Distribution")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 7. Config & Scribble Seed Image

In [ ]:
# ── Prompts ───────────────────────────────────────────────────────────────────
scribble_prompts = [
    "minimalist scribble sketch, single black line drawing, white background, simple doodle",
    "crude stick figure scribble, child's drawing style, black pen on white paper",
    "rough pencil scribble outline, loose sketch, minimal line art",
    "abstract scribble drawing, messy lines, hand-drawn doodle on white",
]

# ── DPS hyperparameters ───────────────────────────────────────────────────────
prompt          = scribble_prompts[2]
negative_prompt = "detailed, realistic, photograph, complex, colored, shading"
guidance_scale  = 7.5
height, width   = 512, 512
n_steps         = 30
num_variations      = N
variation_batch_size = 1
base_zeta_prime = 1.0

print(f"Prompt     : {prompt}")
print(f"Steps      : {n_steps}")
print(f"Variations : {num_variations}/step")
print(f"Zeta base  : {base_zeta_prime}")

In [ ]:
# Generate scribble control image via architect
scribble_latents = architect(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    guidance_scale=guidance_scale,
    output_type="latent",
).images

with torch.no_grad():
    pixels        = architect.vae.decode(scribble_latents.to(torch.float32) / 0.13025).sample
    control_image = architect.image_processor.postprocess(pixels, output_type="pil")[0]

display(control_image)

## 8. Prepare for DPS Loop

In [ ]:
sprinter.vae.to(dtype=torch.float32)

# Gradient checkpointing + freeze all weights
setup_gradient_checkpointing(architect, sprinter)

# ── Encode prompts ────────────────────────────────────────────────────────────
with torch.no_grad():
    (
        prompt_embeds,
        negative_prompt_embeds,
        pooled_prompt_embeds,
        negative_pooled_prompt_embeds,
    ) = architect.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt,
        device=device,
        do_classifier_free_guidance=True,
        num_images_per_prompt=1,
    )

# ── Scheduler + latents ───────────────────────────────────────────────────────
architect.scheduler.set_timesteps(n_steps, device=device)
timesteps = architect.scheduler.timesteps

latents = architect.prepare_latents(
    1, architect.unet.config.in_channels,
    height, width, prompt_embeds.dtype, device, None
)
latents_regular = latents.detach().clone()

# ── SDXL added_cond_kwargs ────────────────────────────────────────────────────
add_time_ids = torch.tensor(
    [[height, width, 0, 0, height, width]], dtype=prompt_embeds.dtype, device=device
)
added_cond_kwargs = {
    "text_embeds" : torch.cat([negative_pooled_prompt_embeds, pooled_prompt_embeds], dim=0),
    "time_ids"    : add_time_ids.repeat(2, 1),
}

# ── CFG encoder states ────────────────────────────────────────────────────────
cfg_encoder_states = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)

# ── Flatten target latents for PCA ───────────────────────────────────────────
all_latents_flat = all_latents.reshape(all_latents.shape[0], -1)

step_gradients = []
step_vis_data  = []

print("✅ Ready for DPS denoising loop.")

## 9. DPS Denoising Loop

In [ ]:
print(f"Beginning DPS Denoising ({n_steps} steps, {num_variations} variations/step)...")

for i, t in enumerate(timesteps):
    print(f"\n{'='*60}")
    print(f"Step {i+1}/{n_steps}  (t={t})")
    print(f"{'='*60}")

    latents_step         = latents.detach().requires_grad_(True)
    latents_step_regular = latents_regular.detach()

    # ── A. Noise prediction (DPS + regular) ───────────────────────────────────
    noise_pred = predict_noise_cfg(
        architect.unet, architect.scheduler,
        latents_step, t, cfg_encoder_states, added_cond_kwargs, guidance_scale
    )
    with torch.no_grad():
        noise_pred_regular = predict_noise_cfg(
            architect.unet, architect.scheduler,
            latents_step_regular, t, cfg_encoder_states, added_cond_kwargs, guidance_scale
        )

    # ── B. pred_x0 ────────────────────────────────────────────────────────────
    pred_x0 = compute_pred_x0(architect.scheduler, noise_pred, t, latents_step)
    with torch.no_grad():
        pred_x0_regular = compute_pred_x0(architect.scheduler, noise_pred_regular, t, latents_step_regular)

    # ── C. Decode pred_x0 → pixel space ──────────────────────────────────────
    pred_x0_scaled = pred_x0 / architect.vae.config.scaling_factor

    def vae_decode_checkpoint(lat):
        return architect.vae.decode(lat.to(architect.vae.dtype)).sample

    pixel_x0      = torch.utils.checkpoint.checkpoint(vae_decode_checkpoint, pred_x0_scaled, use_reentrant=False)
    pixel_x0_norm = torch.clamp((pixel_x0 + 1.0) / 2.0, 0.0, 1.0)

    # ── D. DPS: variations + MMD + gradient ───────────────────────────────────
    grad, mmd_loss, zeta_i, loss_norm, vl_flat = run_dps_step(
        latents, latents_step, noise_pred, pixel_x0_norm,
        sprinter, all_latents, num_variations, variation_batch_size, base_zeta_prime
    )
    grad_norm = grad.norm().item()
    print(f"  MMD={mmd_loss.item():.6f}  ζi={zeta_i:.4f}  ∥∇∥={grad_norm:.6f}")

    step_gradients.append({
        'step'          : i,
        'timestep'      : t.item(),
        'gradient_norm' : grad_norm,
        'mmd_loss'      : mmd_loss.item(),
        'zeta_i'        : zeta_i.item() if isinstance(zeta_i, torch.Tensor) else zeta_i,
        'loss_norm'     : loss_norm.item(),
    })

    # ── E. Store step data + visualize ────────────────────────────────────────
    with torch.no_grad():
        sd = {
            'step'                    : i,
            'timestep'                : t.item(),
            'mmd_loss'                : mmd_loss.item(),
            'zeta_i'                  : zeta_i.item() if isinstance(zeta_i, torch.Tensor) else zeta_i,
            'latents_step_cpu'        : latents_step.detach().cpu(),
            'latents_step_regular_cpu': latents_step_regular.detach().cpu(),
            'pred_x0_cpu'             : pred_x0.detach().cpu(),
            'pred_x0_regular_cpu'     : pred_x0_regular.detach().cpu(),
            'variation_latents_flat'  : vl_flat,
        }
        step_vis_data.append(sd)

    visualize_step(sd, architect, sprinter, all_latents_flat)

    # ── F. Scheduler step (both paths) ───────────────────────────────────────
    correction  = -zeta_i * grad
    latents     = denoise_step(architect.scheduler, noise_pred, t, latents_step, correction=correction)
    with torch.no_grad():
        latents_regular = denoise_step(architect.scheduler, noise_pred_regular, t, latents_step_regular)

    # ── Cleanup ───────────────────────────────────────────────────────────────
    del grad, mmd_loss, loss_norm, zeta_i
    del pixel_x0, pixel_x0_norm, pred_x0, pred_x0_regular
    del latents_step_regular, noise_pred_regular
    gc.collect(); torch.cuda.empty_cache()

del latents_step, noise_pred
torch.cuda.empty_cache()

print(f"\n✅ DPS Complete! {len(step_vis_data)} steps stored.")

## 10. Final Results

In [ ]:
# Decode final latent
with torch.no_grad():
    final_image = latent_to_pil(latents.cpu().to(device), architect.vae, architect.image_processor)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(base_image_pil); axes[0].set_title("Input Shape");  axes[0].axis('off')
axes[1].imshow(final_image);    axes[1].set_title("DPS Output");   axes[1].axis('off')
plt.suptitle("DPS Result", fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Training curves
steps      = [d['step']          for d in step_gradients]
mmd_vals   = [d['mmd_loss']      for d in step_gradients]
grad_norms = [d['gradient_norm'] for d in step_gradients]
zetas      = [d['zeta_i']        for d in step_gradients]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("DPS Training Curves", fontsize=14, fontweight='bold')

axes[0].plot(steps, mmd_vals,   color='royalblue');  axes[0].set_title("MMD Loss");      axes[0].set_xlabel("Step"); axes[0].grid(True, alpha=0.3)
axes[1].plot(steps, grad_norms, color='crimson');    axes[1].set_title("Gradient Norm"); axes[1].set_xlabel("Step"); axes[1].grid(True, alpha=0.3)
axes[2].plot(steps, zetas,      color='seagreen');   axes[2].set_title("Zeta (ζ)");      axes[2].set_xlabel("Step"); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()